In [1]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple


# ============================================================
# CONFIG
# ============================================================

@dataclass
class ResearchConfig:
    positionLimit: int = 80
    markoutHorizons: Tuple[int, ...] = (1, 2, 5, 10, 20)
    cycleMin: int = 3
    cycleMax: int = 200
    localWindowList: Tuple[int, ...] = (5, 10, 20, 40)
    breakoutWindowList: Tuple[int, ...] = (5, 10, 20, 40)
    stateBuckets: int = 5


# ============================================================
# FEATURE ENGINEERING
# ============================================================

def prepareBookFeatures(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()

    required = ["timestamp", "bestBid", "bestAsk", "bidVol", "askVol"]
    missing = [col for col in required if col not in data.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    data = data.sort_values("timestamp").reset_index(drop=True)

    data["mid"] = (data["bestBid"] + data["bestAsk"]) / 2.0
    data["spread"] = data["bestAsk"] - data["bestBid"]

    totalVol = data["bidVol"].abs() + data["askVol"].abs()
    totalVol = totalVol.replace(0, np.nan)

    data["microprice"] = (
        data["bestBid"] * data["askVol"].abs() + data["bestAsk"] * data["bidVol"].abs()
    ) / totalVol
    data["microprice"] = data["microprice"].fillna(data["mid"])

    data["imbalance"] = (
        data["bidVol"].abs() - data["askVol"].abs()
    ) / totalVol
    data["imbalance"] = data["imbalance"].fillna(0.0)

    data["ret1"] = data["mid"].diff()
    data["ret2"] = data["mid"].diff(2)
    data["ret5"] = data["mid"].diff(5)
    data["ret10"] = data["mid"].diff(10)

    data["microMinusMid"] = data["microprice"] - data["mid"]

    data["spreadChange"] = data["spread"].diff()
    data["imbalanceChange"] = data["imbalance"].diff()

    data["rollingVol10"] = data["ret1"].rolling(10).std()
    data["rollingVol25"] = data["ret1"].rolling(25).std()
    data["rollingVol50"] = data["ret1"].rolling(50).std()

    if "lastTradePrice" in data.columns:
        data["tradeToMid"] = data["lastTradePrice"] - data["mid"]
    else:
        data["tradeToMid"] = np.nan

    if "lastTradeQty" in data.columns:
        data["signedTradeQty"] = data["lastTradeQty"]
    else:
        data["signedTradeQty"] = np.nan

    return data


def addForwardReturns(df: pd.DataFrame, horizons: Tuple[int, ...]) -> pd.DataFrame:
    data = df.copy()
    for h in horizons:
        data[f"fwdRet_{h}"] = data["mid"].shift(-h) - data["mid"]
    return data


# ============================================================
# BASIC DIAGNOSTICS
# ============================================================

def autocorrelationScan(series: pd.Series, maxLag: int = 50) -> pd.DataFrame:
    clean = series.dropna()
    rows = []
    for lag in range(1, maxLag + 1):
        if len(clean) <= lag:
            break
        rows.append({
            "lag": lag,
            "autocorr": clean.autocorr(lag=lag)
        })
    return pd.DataFrame(rows)


def cycleScan(series: pd.Series, cycleMin: int = 3, cycleMax: int = 200) -> pd.DataFrame:
    clean = series.reset_index(drop=True)
    rows = []

    for k in range(cycleMin, cycleMax + 1):
        if len(clean) < 5 * k:
            continue

        modGroups = []
        for i in range(k):
            vals = clean[clean.index % k == i].dropna()
            if len(vals) > 0:
                modGroups.append(vals.mean())

        if len(modGroups) < 2:
            continue

        strength = np.std(modGroups)
        rows.append({
            "cycleLength": k,
            "modMeanStd": strength,
            "maxModMean": np.max(modGroups),
            "minModMean": np.min(modGroups),
            "rangeModMean": np.max(modGroups) - np.min(modGroups)
        })

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["modMeanStd", "rangeModMean"], ascending=False).reset_index(drop=True)
    return out


def fftScan(series: pd.Series, topN: int = 20) -> pd.DataFrame:
    clean = series.dropna().values
    if len(clean) < 16:
        return pd.DataFrame()

    demeaned = clean - np.mean(clean)
    fftVals = np.fft.rfft(demeaned)
    power = np.abs(fftVals) ** 2
    freqs = np.fft.rfftfreq(len(demeaned), d=1.0)

    rows = []
    for freq, pwr in zip(freqs[1:], power[1:]):  # skip zero frequency
        if freq <= 0:
            continue
        period = 1.0 / freq
        rows.append({
            "frequency": freq,
            "period": period,
            "power": pwr
        })

    out = pd.DataFrame(rows).sort_values("power", ascending=False).head(topN).reset_index(drop=True)
    return out


# ============================================================
# CONDITIONAL NEXT-RETURN TESTS
# ============================================================

def bucketedConditionalReturns(
    df: pd.DataFrame,
    featureCol: str,
    forwardCol: str,
    numBuckets: int = 5
) -> pd.DataFrame:
    data = df[[featureCol, forwardCol]].dropna().copy()
    if len(data) < 50:
        return pd.DataFrame()

    try:
        data["bucket"] = pd.qcut(data[featureCol], q=numBuckets, duplicates="drop")
    except Exception:
        return pd.DataFrame()

    out = (
        data.groupby("bucket", observed=False)[forwardCol]
        .agg(["count", "mean", "std"])
        .reset_index()
        .rename(columns={"mean": "avgForwardReturn", "std": "stdForwardReturn"})
    )

    hitRates = (
        data.assign(hit=(data[forwardCol] > 0).astype(int))
        .groupby("bucket", observed=False)["hit"]
        .mean()
        .reset_index(name="upMoveHitRate")
    )

    out = out.merge(hitRates, on="bucket", how="left")
    out["feature"] = featureCol
    out["forward"] = forwardCol
    return out


def runStateTests(df: pd.DataFrame, horizons: Tuple[int, ...], numBuckets: int = 5) -> Dict[str, pd.DataFrame]:
    features = [
        "imbalance",
        "microMinusMid",
        "spread",
        "spreadChange",
        "imbalanceChange",
        "ret1",
        "ret2",
        "ret5",
        "ret10",
        "rollingVol10",
        "rollingVol25",
    ]

    results = {}
    for feature in features:
        if feature not in df.columns:
            continue
        frames = []
        for h in horizons:
            fwd = f"fwdRet_{h}"
            if fwd in df.columns:
                table = bucketedConditionalReturns(df, feature, fwd, numBuckets=numBuckets)
                if not table.empty:
                    frames.append(table)
        if frames:
            results[feature] = pd.concat(frames, ignore_index=True)
    return results


# ============================================================
# LOCAL EXTREMA / BREAKOUT TESTS
# ============================================================

def localExtremaTests(df: pd.DataFrame, windows: Tuple[int, ...], horizons: Tuple[int, ...]) -> pd.DataFrame:
    rows = []
    for w in windows:
        rollingMax = df["mid"].rolling(w).max()
        rollingMin = df["mid"].rolling(w).min()

        isLocalHigh = df["mid"] >= rollingMax
        isLocalLow = df["mid"] <= rollingMin

        for h in horizons:
            fwd = f"fwdRet_{h}"
            if fwd not in df.columns:
                continue

            highVals = df.loc[isLocalHigh, fwd].dropna()
            lowVals = df.loc[isLocalLow, fwd].dropna()

            rows.append({
                "window": w,
                "horizon": h,
                "event": "localHigh",
                "count": len(highVals),
                "avgForwardReturn": highVals.mean() if len(highVals) else np.nan,
                "upMoveHitRate": (highVals > 0).mean() if len(highVals) else np.nan
            })

            rows.append({
                "window": w,
                "horizon": h,
                "event": "localLow",
                "count": len(lowVals),
                "avgForwardReturn": lowVals.mean() if len(lowVals) else np.nan,
                "upMoveHitRate": (lowVals > 0).mean() if len(lowVals) else np.nan
            })

    return pd.DataFrame(rows)


def breakoutTests(df: pd.DataFrame, windows: Tuple[int, ...], horizons: Tuple[int, ...]) -> pd.DataFrame:
    rows = []
    for w in windows:
        prevMax = df["mid"].shift(1).rolling(w).max()
        prevMin = df["mid"].shift(1).rolling(w).min()

        breakoutUp = df["mid"] > prevMax
        breakoutDown = df["mid"] < prevMin

        for h in horizons:
            fwd = f"fwdRet_{h}"
            if fwd not in df.columns:
                continue

            upVals = df.loc[breakoutUp, fwd].dropna()
            downVals = df.loc[breakoutDown, fwd].dropna()

            rows.append({
                "window": w,
                "horizon": h,
                "event": "breakoutUp",
                "count": len(upVals),
                "avgForwardReturn": upVals.mean() if len(upVals) else np.nan,
                "upMoveHitRate": (upVals > 0).mean() if len(upVals) else np.nan
            })

            rows.append({
                "window": w,
                "horizon": h,
                "event": "breakoutDown",
                "count": len(downVals),
                "avgForwardReturn": downVals.mean() if len(downVals) else np.nan,
                "upMoveHitRate": (downVals > 0).mean() if len(downVals) else np.nan
            })

    return pd.DataFrame(rows)


# ============================================================
# SIMPLE MM MARKOUT TESTS
# ============================================================

def simulatePassiveFillCandidates(
    df: pd.DataFrame,
    quoteOffset: int = 1
) -> pd.DataFrame:
    """
    Crude markout proxy:
    - Pretend we quote bid = bestBid + quoteOffset and ask = bestAsk - quoteOffset
      if those remain valid.
    - Fill assumption is simplified:
        buy candidate if next mid <= our bid or best ask trades through
        sell candidate if next mid >= our ask or best bid trades through

    This is not a production fill simulator.
    It is a research proxy for whether quoting states are good or bad.
    """
    data = df.copy()

    data["simBid"] = np.minimum(data["bestBid"] + quoteOffset, data["bestAsk"] - 1)
    data["simAsk"] = np.maximum(data["bestAsk"] - quoteOffset, data["bestBid"] + 1)

    nextMid = data["mid"].shift(-1)
    data["buyFillProxy"] = (nextMid <= data["simBid"]).astype(int)
    data["sellFillProxy"] = (nextMid >= data["simAsk"]).astype(int)

    return data


def computeFillMarkouts(
    df: pd.DataFrame,
    horizons: Tuple[int, ...]
) -> pd.DataFrame:
    rows = []
    for side, fillCol, pxCol, sign in [
        ("BUY", "buyFillProxy", "simBid", 1.0),
        ("SELL", "sellFillProxy", "simAsk", -1.0),
    ]:
        fills = df[df[fillCol] == 1].copy()
        if fills.empty:
            continue

        for h in horizons:
            futureMid = fills["mid"].shift(-h)
            markout = sign * (futureMid - fills[pxCol])

            rows.append({
                "side": side,
                "horizon": h,
                "count": markout.notna().sum(),
                "avgMarkout": markout.mean(),
                "medianMarkout": markout.median(),
                "positiveMarkoutRate": (markout > 0).mean()
            })

    return pd.DataFrame(rows)


def regimeGatedMarkouts(
    df: pd.DataFrame,
    featureCol: str,
    horizons: Tuple[int, ...],
    numBuckets: int = 5
) -> pd.DataFrame:
    data = df.copy()
    if featureCol not in data.columns:
        return pd.DataFrame()

    try:
        data["bucket"] = pd.qcut(data[featureCol], q=numBuckets, duplicates="drop")
    except Exception:
        return pd.DataFrame()

    rows = []
    for bucket, sub in data.groupby("bucket", observed=False):
        for h in horizons:
            buyMask = sub["buyFillProxy"] == 1
            sellMask = sub["sellFillProxy"] == 1

            buyFuture = sub["mid"].shift(-h)
            sellFuture = sub["mid"].shift(-h)

            buyMarkout = buyFuture[buyMask] - sub.loc[buyMask, "simBid"]
            sellMarkout = sub.loc[sellMask, "simAsk"] - sellFuture[sellMask]

            rows.append({
                "feature": featureCol,
                "bucket": str(bucket),
                "horizon": h,
                "buyCount": buyMarkout.notna().sum(),
                "buyAvgMarkout": buyMarkout.mean(),
                "sellCount": sellMarkout.notna().sum(),
                "sellAvgMarkout": sellMarkout.mean()
            })

    return pd.DataFrame(rows)


# ============================================================
# SUMMARY SCORING
# ============================================================

def summarizeBestSignals(stateTests: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for feature, table in stateTests.items():
        if table.empty:
            continue

        temp = table.copy()
        temp["absAvgForwardReturn"] = temp["avgForwardReturn"].abs()

        best = temp.sort_values(["absAvgForwardReturn", "count"], ascending=[False, False]).iloc[0]
        rows.append({
            "feature": feature,
            "forward": best["forward"],
            "bucket": str(best["bucket"]),
            "count": best["count"],
            "avgForwardReturn": best["avgForwardReturn"],
            "upMoveHitRate": best["upMoveHitRate"]
        })

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values("avgForwardReturn", key=lambda s: s.abs(), ascending=False).reset_index(drop=True)
    return out


# ============================================================
# MAIN ENTRY POINT
# ============================================================

def runAshResearch(dfRaw: pd.DataFrame, config: Optional[ResearchConfig] = None) -> Dict[str, pd.DataFrame]:
    if config is None:
        config = ResearchConfig()

    df = prepareBookFeatures(dfRaw)
    df = addForwardReturns(df, config.markoutHorizons)

    autoRet = autocorrelationScan(df["ret1"], maxLag=50)
    autoImb = autocorrelationScan(df["imbalance"], maxLag=50)

    cycleReturns = cycleScan(df["ret1"], cycleMin=config.cycleMin, cycleMax=config.cycleMax)
    cycleMid = cycleScan(df["mid"], cycleMin=config.cycleMin, cycleMax=config.cycleMax)

    fftReturns = fftScan(df["ret1"], topN=20)

    stateTests = runStateTests(df, config.markoutHorizons, numBuckets=config.stateBuckets)
    bestSignals = summarizeBestSignals(stateTests)

    extrema = localExtremaTests(df, config.localWindowList, config.markoutHorizons)
    breakouts = breakoutTests(df, config.breakoutWindowList, config.markoutHorizons)

    sim = simulatePassiveFillCandidates(df, quoteOffset=1)
    fillMarkouts = computeFillMarkouts(sim, config.markoutHorizons)

    regimeTables = []
    for feature in ["imbalance", "microMinusMid", "spread", "rollingVol10", "ret1", "ret5"]:
        table = regimeGatedMarkouts(sim, feature, config.markoutHorizons, numBuckets=config.stateBuckets)
        if not table.empty:
            regimeTables.append(table)

    regimeMarkouts = pd.concat(regimeTables, ignore_index=True) if regimeTables else pd.DataFrame()

    return {
        "features": df,
        "autocorr_returns": autoRet,
        "autocorr_imbalance": autoImb,
        "cycle_returns": cycleReturns,
        "cycle_mid": cycleMid,
        "fft_returns": fftReturns,
        "state_tests_best": bestSignals,
        "extrema_tests": extrema,
        "breakout_tests": breakouts,
        "fill_markouts": fillMarkouts,
        "regime_markouts": regimeMarkouts,
        **{f"state_{k}": v for k, v in stateTests.items()}
    }

In [2]:
import pandas as pd

ashDf = pd.read_csv("ash_round1.csv")

results = runAshResearch(ashDf)

print("\nAUTOCORR RETURNS")
print(results["autocorr_returns"].head(15))

print("\nTOP CYCLE CANDIDATES")
print(results["cycle_returns"].head(15))

print("\nTOP FFT PERIODS")
print(results["fft_returns"].head(15))

print("\nBEST STATE SIGNALS")
print(results["state_tests_best"].head(20))

print("\nEXTREMA TESTS")
print(results["extrema_tests"].sort_values("avgForwardReturn", key=lambda s: s.abs(), ascending=False).head(20))

print("\nBREAKOUT TESTS")
print(results["breakout_tests"].sort_values("avgForwardReturn", key=lambda s: s.abs(), ascending=False).head(20))

print("\nFILL MARKOUTS")
print(results["fill_markouts"])

print("\nREGIME MARKOUTS")
print(results["regime_markouts"].head(30))

FileNotFoundError: [Errno 2] No such file or directory: 'ash_round1.csv'